# Linnet in five minutes

Write a model, let the checker catch a shape error, then run and serve a
Hugging Face model through Linnet. On Colab, pick a GPU runtime first
(Runtime, then Change runtime type).

In [ ]:
%pip install -q "linnet-lang[serve]"
!linnet --version

## Write and check a model

A `.linnet` file declares every parameter's shape and dtype. Dimensions are
generics, and the compiler proves every shape before anything runs.

In [ ]:
%%writefile mlp.linnet
module mlp

use std.nn.activations::{relu}
use std.nn.linear::{Linear}

pub block Mlp<In: Dim, Hidden: Dim, Out: Dim, T: Float = f32> {
    sub up: Linear<In, Hidden, T>
    sub down: Linear<Hidden, Out, T>

    pub entry forward<B: Dim>(x: Tensor[B, In; T]) -> Tensor[B, Out; T] {
        return down.forward(relu(up.forward(x)))
    }
}

In [ ]:
!linnet check mlp.linnet && linnet inspect --parameters mlp.linnet

Splitting heads with a `reshape` needs `Heads > 0` and
`Hidden % Heads == 0`. Leave the `where` clause out and the checker names
what is missing:

In [ ]:
%%writefile heads.linnet
module heads

fn split<B: Dim, S: Dim, Hidden: Dim, Heads: Dim>(
    x: Tensor[B, S, Hidden; f32],
) -> Tensor[B, S, Heads, Hidden / Heads; f32] {
    return reshape(x, [B, S, Heads, Hidden / Heads])
}

In [ ]:
!linnet check heads.linnet

The same file loads as a `torch.nn.Module`:

In [ ]:
import torch
from linnet.torch import load

mlp = load("mlp.linnet", generics={"In": 4, "Hidden": 8, "Out": 2})
mlp(torch.randn(3, 4)).shape

## Run a Hugging Face model

`linnet.nest.load` takes a model from [Nest](https://nest.franknoh.dev), a
Hub repo, or a directory. A `transformers` checkpoint of a family Nest knows
(Llama, Mistral, Qwen2, Qwen3, Phi-3, GPT-2) is converted to Linnet on the
way and checked against the checkpoint's tensors. No code from the repo
runs.

In [ ]:
from transformers import AutoTokenizer

from linnet import nest
from linnet.serve import Engine, Request

name = "Qwen/Qwen2.5-0.5B-Instruct"
dtype = "bf16" if torch.cuda.is_bf16_supported() else "f16"
model = nest.load(
    name,
    device="cuda",
    numerics="fast",
    compile=True,
    generics={"Batch": 4, "MaxSeq": 1024, "T": dtype},
    cast_dtype=True,
)
tokenizer = AutoTokenizer.from_pretrained(name)
engine = Engine(model)

In [ ]:
messages = [{"role": "user", "content": "In one sentence: why check tensor shapes before running?"}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
prompt = tokenizer.encode(text, add_special_tokens=False)
done, stats = engine.run(
    [Request(prompt=prompt, max_new_tokens=96, eos=frozenset({tokenizer.eos_token_id}))]
)
print(tokenizer.decode(done[0].tokens, skip_special_tokens=True))

## Serve it with OpenAI's API

`linnet serve` takes the same names. It batches requests continuously and
replays each step as a CUDA graph.

In [ ]:
import json
import subprocess
import time
import urllib.request

del model, engine
torch.cuda.empty_cache()
server = subprocess.Popen(["linnet", "serve", name, "--batch", "8", "--max-seq", "2048"])
for _ in range(900):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/health")
        break
    except OSError:
        time.sleep(1)

In [ ]:
body = {
    "model": name,
    "messages": [{"role": "user", "content": "Name three primary colors."}],
    "max_tokens": 32,
}
request = urllib.request.Request(
    "http://127.0.0.1:8000/v1/chat/completions",
    json.dumps(body).encode(),
    {"Content-Type": "application/json"},
)
print(json.load(urllib.request.urlopen(request))["choices"][0]["message"]["content"])

In [ ]:
server.terminate()

## Next

- [The whole language](https://linnet.franknoh.dev/docs/language) in one page
- [Nest](https://nest.franknoh.dev): 24 checked models
- [Training](https://linnet.franknoh.dev/docs/training): SFT, LoRA, FSDP, DPO and GRPO
- [Benchmarks](https://linnet.franknoh.dev/benchmarks)